# ExecPulse-BI: Automated Data Modeling Pipeline

This notebook simulates messy raw transactional data, cleans it using `pandas`, and restructures it into a strict **Star Schema** (`Fact_Sales`, `Dim_Customer`, `Dim_Product`, `Dim_Date`).

Outputs are saved as CSVs and a local `SQLite` database for direct BI ingestion.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime, timedelta
import random

# Ensure output directories exist
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Synthesize Dimension Data (Products & Customers)

In [ ]:
np.random.seed(42)
random.seed(42)

# --- Dim_Product ---
categories = {
    'Electronics': ['Laptops', 'Smartphones', 'Audio', 'Accessories'],
    'Furniture': ['Chairs', 'Desks', 'Tables', 'Storage'],
    'Apparel': ['Mens', 'Womens', 'Kids', 'Shoes']
}

products = []
for pid in range(1, 151):
    cat = random.choice(list(categories.keys()))
    subcat = random.choice(categories[cat])
    unit_cost = round(random.uniform(10.0, 500.0), 2)
    unit_price = round(unit_cost * random.uniform(1.2, 2.5), 2)
    products.append({
        'product_id': f"PRD-{pid:04d}",
        'product_name': f"{subcat} Item {pid}",
        'category': cat,
        'sub_category': subcat,
        'unit_price': unit_price,
        'unit_cost': unit_cost
    })

dim_product = pd.DataFrame(products)
print(f"Created Dim_Product: {dim_product.shape[0]} records")
dim_product.head(3)

In [ ]:
# --- Dim_Customer ---
regions = ['North America', 'EMEA', 'APAC', 'LATAM']
segments = ['Consumer', 'Corporate', 'Home Office']
countries = ['US', 'Canada', 'UK', 'Germany', 'Australia', 'Japan', 'Brazil']

customers = []
for cid in range(1, 1001):
    customers.append({
        'customer_id': f"CUST-{cid:05d}",
        'first_name': f"FirstName_{cid}",
        'last_name': f"LastName_{cid}",
        'segment': random.choice(segments),
        'region': random.choice(regions),
        'country': random.choice(countries)
    })

dim_customer = pd.DataFrame(customers)
print(f"Created Dim_Customer: {dim_customer.shape[0]} records")
dim_customer.head(3)

## 2. Synthesize Date Dimension

In [ ]:
# --- Dim_Date ---
start_date = datetime(2022, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = pd.date_range(start=start_date, end=end_date)

dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_of_week'] = dim_date['full_date'].dt.day_name()

# Reorder columns putting date_key first
dim_date = dim_date[['date_key', 'full_date', 'year', 'quarter', 'month', 'month_name', 'day', 'day_of_week']]
print(f"Created Dim_Date: {dim_date.shape[0]} records")
dim_date.head(3)

## 3. Synthesize Fact Table (Sales)

In [ ]:
# --- Fact_Sales ---
num_transactions = 50000  # Synthesize 50k transactions

# Randomly sample IDs from our dimensions
sampled_products = dim_product.sample(num_transactions, replace=True).reset_index(drop=True)
sampled_customers = dim_customer.sample(num_transactions, replace=True).reset_index(drop=True)
sampled_dates = dim_date.sample(num_transactions, replace=True).reset_index(drop=True)

sales = pd.DataFrame({
    'sale_id': [f"TXN-{i:07d}" for i in range(1, num_transactions + 1)],
    'date_key': sampled_dates['date_key'],
    'customer_id': sampled_customers['customer_id'],
    'product_id': sampled_products['product_id'],
    'quantity': np.random.randint(1, 10, size=num_transactions),
})

# Calculate financials logically
sales['unit_price'] = sampled_products['unit_price']
sales['discount_amount'] = round(sales['unit_price'] * sales['quantity'] * np.random.uniform(0, 0.2, size=num_transactions), 2)
sales['total_amount'] = round((sales['unit_price'] * sales['quantity']) - sales['discount_amount'], 2)
sales['tax_amount'] = round(sales['total_amount'] * 0.07, 2) # 7% assumed tax rate

# Drop unit price since it lives in Dim_Product (strictly normalized star schema)
sales = sales.drop(columns=['unit_price'])

fact_sales = sales
print(f"Created Fact_Sales: {fact_sales.shape[0]} records")
fact_sales.head(3)

## 4. Export to Processed Formats (CSV & SQLite)

In [ ]:
# Export as CSVs for BI tools that prefer flat files
dim_product.to_csv(os.path.join(PROCESSED_DIR, 'Dim_Product.csv'), index=False)
dim_customer.to_csv(os.path.join(PROCESSED_DIR, 'Dim_Customer.csv'), index=False)
dim_date.to_csv(os.path.join(PROCESSED_DIR, 'Dim_Date.csv'), index=False)
fact_sales.to_csv(os.path.join(PROCESSED_DIR, 'Fact_Sales.csv'), index=False)

print("✅ CSV files successfully exported to data/processed/")

In [ ]:
# Convert full_date column to string format for SQLite compatibility
dim_date['full_date'] = dim_date['full_date'].dt.strftime('%Y-%m-%d')

# Export to SQLite database (Unified BI source)
db_path = os.path.join(PROCESSED_DIR, 'execpulse_star_schema.db')
conn = sqlite3.connect(db_path)

try:
    dim_product.to_sql('Dim_Product', conn, if_exists='replace', index=False)
    dim_customer.to_sql('Dim_Customer', conn, if_exists='replace', index=False)
    dim_date.to_sql('Dim_Date', conn, if_exists='replace', index=False)
    fact_sales.to_sql('Fact_Sales', conn, if_exists='replace', index=False)
    print(f"✅ Star schema successfully loaded into SQLite database: {db_path}")
except Exception as e:
    print(f"❌ Error writing to SQLite: {e}")
finally:
    conn.close()

## 5. Next Steps
1. Open **Power BI / Tableau**.
2. Connect to the generated SQLite database (`execpulse_star_schema.db`) or the folder containing the CSVs.
3. Set up Relationships mapping the 1-to-Many dimension constraints to the Fact table.
4. Construct the required DAX/LOD measures for KPIs (Refer to `docs/usage/README.md`).